
# CustomerInsight AI — Clustering Stability Research

**Phase 05: Reliability of Customer Segmentation**

## Research objective

We previously found that K-Means had the highest
silhouette score among our baseline models.

However, a good silhouette score does not guarantee
that customer group assignments are stable.

In this notebook, we investigate:

1. Sensitivity to random model initialization.
2. Sensitivity to changes in the training sample.
3. Sensitivity to small amounts of feature noise.

We use the Adjusted Rand Index (ARI) to compare
cluster assignments.

An ARI of 1 indicates perfect assignment agreement.
Values near 0 indicate agreement around chance level.

The results are specific to our tested conditions.


In [5]:

from pathlib import Path

import pandas as pd
import plotly.express as px

from customerinsight.clustering import prepare_clustering_features
from customerinsight.stability import run_stability_study

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/customer_rfm.csv"

rfm = pd.read_csv(DATA_PATH, dtype={"customer_id": "string"})
features = prepare_clustering_features(rfm)

print(f"Customers: {len(rfm):,}")
print("Features used:", features.columns.tolist())

display(rfm.head())


Customers: 4,338
Features used: ['recency', 'frequency', 'monetary']


,customer_id,recency,frequency,monetary
0,12346,326,1,77183.60
1,12347,3,7,4310.00
2,12348,76,4,1797.24
3,12349,19,1,1757.55
4,12350,311,1,334.40



## Experimental methodology

We conduct 20 runs for each experiment.

**Random initialization**

Train K-Means with different random seeds using
the same customer data.

**Training subset**

Train K-Means using different random 80% subsets.
Then predict the same full set of customers.

**Feature noise**

Add small Gaussian noise to standardized customer
features, retrain K-Means and compare the resulting
assignments with the original baseline.

### Important limitations

The experiments evaluate consistency with a baseline,
not correctness against known customer categories.

Feature preprocessing is held fixed in this study.
We do not yet evaluate the stability of the entire
data preprocessing and clustering pipeline.


In [6]:

# Run repeatable stability experiments using our reusable module.
detailed, summary = run_stability_study(features, runs=20)

print("Stability experiment summary:")
display(summary)

print("Individual experiment runs:")
display(detailed.head(10))

# Preserve the numerical results locally for documentation.
output_dir = PROJECT_ROOT / "results/experiments"
output_dir.mkdir(parents=True, exist_ok=True)

detailed.to_csv(output_dir / "stability_runs.csv", index=False)
summary.to_csv(output_dir / "stability_summary.csv", index=False)

print("Experiment results saved.")


c:\Users\x03xN\Documents\customerinsight-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\x03xN\Documents\customerinsight-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\x03xN\Documents\customerinsight-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\x03xN\Documents\customerinsight-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\x03xN\Documents\customerinsight-ai\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMea

Stability experiment summary:


,experiment,mean,std,min,max
0,Feature noise,0.9172,0.0200,0.8756,0.9408
1,Random initialization,0.9360,0.0298,0.9132,1.0000
2,Training subset,0.9413,0.0381,0.8840,0.9942


Individual experiment runs:


,experiment,run,ari
0,Random initialization,1,0.920180
1,Random initialization,2,0.917132
2,Random initialization,3,0.913702
3,Random initialization,4,0.913702
4,Random initialization,5,0.913225
5,Random initialization,6,0.946684
6,Random initialization,7,0.948488
7,Random initialization,8,0.913702
8,Random initialization,9,0.953038
9,Random initialization,10,0.996166


Experiment results saved.



## Visualization 1 — Which experimental condition is most stable?

### Why are we using a bar chart?

We want to compare the average assignment agreement
across three different experimental conditions.

A bar chart allows us to compare these values easily.

### How to read this chart

Higher ARI indicates greater agreement with the
original clustering assignments.

However, an average hides variation between runs.

We therefore inspect the distribution of individual
ARI values in the next chart.


In [7]:

fig = px.bar(
    summary,
    x="experiment",
    y="mean",
    text="mean",
    title="How stable are the customer groups?",
    labels={
        "experiment": "Experimental condition",
        "mean": "Average Adjusted Rand Index",
    },
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)

fig.update_layout(
    yaxis_title="Average assignment agreement",
)

fig.show()



## Visualization 2 — Do the results vary between runs?

### Why use a box plot?

Two experiments can have similar averages but
different levels of variation.

A box plot shows the median, spread and unusually
high or low outcomes.

### How to interpret it

A narrow box suggests the ARI values across
repeated experiments are relatively consistent.

A wide box suggests the results vary more.

We should not interpret low variability as proof
that the identified customer segments are useful.


In [8]:

fig = px.box(
    detailed,
    x="experiment",
    y="ari",
    color="experiment",
    points="all",
    title="How much does cluster stability vary between runs?",
    labels={
        "experiment": "Experimental condition",
        "ari": "Adjusted Rand Index",
    },
)

fig.update_layout(showlegend=False)
fig.show()



## Research findings

Complete this section using the actual results.

### Random initialization

Record the mean ARI and whether different random
seeds produced similar assignments.

### Training subset

Record how much assignments changed when models
were trained on different customer samples.

### Feature noise

Describe the impact of small perturbations on
customer grouping consistency.

### Interpretation

Which condition produced the lowest average ARI?

Which experiment showed the widest variation?

Were any results unexpectedly stable or unstable?

### Limitations

This study evaluates K-Means using one dataset
and a limited set of experimental conditions.

We used fixed feature preprocessing.

The comparison uses a full-data baseline rather
than independently verified ground-truth labels.

Further work is needed to assess other algorithms,
noise levels, sample fractions and business meaning.
